# Fitness app onboarding: activity over four weeks

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Two-way Repeated Measures Analysis of Variance** · a worked example from the StatNex teaching guide

**The story.** A fitness app tested a new onboarding flow. Six users got the old onboarding and six the new one, and each user's daily active minutes were recorded at weeks 0, 1, 2 and 3.

**The question.** Does the new onboarding keep users active for longer?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/two_way_anova_repeated.csv` (12 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
User No.,Onboarding,Week 0,Week 1,Week 2,Week 3
1.0,Old,10.5,13.1,9.6,4.8
2.0,Old,9.5,10.7,7.0,2.6
3.0,Old,10.6,12.2,7.9,4.1
4.0,Old,12.1,13.9,10.3,5.8
5.0,Old,13.5,13.4,11.1,6.7
6.0,Old,12.5,12.9,9.2,5.3
7.0,New,7.5,7.5,7.0,6.1
8.0,New,13.1,12.8,11.1,10.5
9.0,New,10.0,9.6,10.2,7.5
10.0,New,12.9,13.5,13.1,10.6
11.0,New,11.0,11.0,9.9,7.8
12.0,New,11.8,12.1,11.1,8.5

These cells save `eta_squared_interval.py`, `sphericity.py`: steps that no library does, written out. The analysis imports them.

In [ ]:
%%writefile eta_squared_interval.py
"""The confidence interval for partial eta squared, from the noncentral F distribution.

SciPy has the noncentral F (stats.ncf) but no function that inverts it for an interval,
so the inversion is written out (Steiger 2004). The interval is found for the
noncentrality λ and converted to partial eta squared as λ / (λ + df1 + df2 + 1).
"""

import numpy as np
from scipy import optimize, stats


def _limit(f, df1, df2, q):
    """The λ ≥ 0 under which the observed F sits at the q quantile of F(df1, df2, λ).

    0 when even λ = 0 cannot put it there: an F too small for the interval to leave 0.
    The cdf at F falls as λ rises, so the bracket opens outwards from F·df1, doubling.
    """
    def gap(lam):
        cdf = stats.f.cdf(f, df1, df2) if lam == 0 else stats.ncf.cdf(f, df1, df2, lam)
        return cdf - q

    if gap(0.0) <= 0:
        return 0.0
    start = max(f * df1, 1.0)
    step = 1.0
    while gap(max(start - step, 0.0)) <= 0:
        step *= 2
    low = max(start - step, 0.0)
    step = 1.0
    while gap(start + step) >= 0:
        step *= 2
    return optimize.brentq(gap, low, start + step, xtol=1e-14,
                           rtol=4 * np.finfo(float).eps, maxiter=500)


def partial_eta_squared_interval(f, df1, df2, level):
    """(lower, upper): the level interval for partial eta squared (Steiger 2004)."""
    total = df1 + df2 + 1
    lower = _limit(f, df1, df2, (1 + level) / 2)
    upper = _limit(f, df1, df2, (1 - level) / 2)
    return lower / (lower + total), upper / (upper + total)

In [ ]:
%%writefile sphericity.py
"""Mauchly's test of sphericity and the epsilon corrections for repeated measures.

statsmodels' AnovaRM prints only the uncorrected F test and SciPy has neither, so both
are written out. Each takes the covariance matrix of one within-subject effect's
orthonormal contrast scores -- the residuals of the between-subject model when there is
one -- which is all that sphericity is about.
"""

import numpy as np
from scipy import stats


def mauchly(covariance, df):
    """Mauchly's W, its chi-square approximation, the chi-square df and P.

    ``df`` is the error df of the covariance matrix (subjects minus groups). W is 1 when
    the contrasts are uncorrelated with equal variances, which is what sphericity means.
    With one contrast -- a two-level factor -- there is nothing to test, so W = 1 and P
    is undefined.
    """
    p = covariance.shape[0]
    if p < 2:
        return 1.0, 0.0, 0, np.nan
    w = np.linalg.det(covariance) / (np.trace(covariance) / p) ** p
    correction = 1 - (2 * p**2 + p + 2) / (6 * p * df)
    chi_square = -df * correction * np.log(w)
    chi_df = p * (p + 1) // 2 - 1
    return w, chi_square, chi_df, stats.chi2.sf(chi_square, chi_df)


def epsilons(covariance, n, groups):
    """(lower bound, Greenhouse-Geisser, Huynh-Feldt-Lecoutre, Chi-Muller) for one effect.

    Epsilon is 1 under sphericity and 1/p at worst. Huynh-Feldt's estimate uses
    Lecoutre's (1991) correction, (N - g + 1) in the numerator, and can exceed 1; the
    table caps it at 1 when it multiplies the degrees of freedom. Chi-Muller shrinks it,
    since it runs high in small samples, by a multiple fixed by the design alone,
    (m - 1)(m - 2) / m^2 with m = p (N - g - 1), and floors it at the lower bound
    (Chi et al. 2012; SAS's UEPSDEF=CM).
    """
    p = covariance.shape[0]
    lower = 1 / p
    gg = np.trace(covariance) ** 2 / (p * np.trace(covariance @ covariance))
    gg = min(max(gg, lower), 1.0)  # rounding can step a hair outside [1/p, 1]
    df = n - groups
    hfl = ((df + 1) * p * gg - 2) / (p * (df - p * gg))
    m = p * (df - 1)
    cm = max(hfl * (m - 1) * (m - 2) / m**2, lower) if m > 0 else lower
    return lower, gg, hfl, cm

## Libraries

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from scipy.linalg import helmert
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests
from sphericity import epsilons, mauchly

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per subject: a column per condition (the within-subject factor) and the
# group the subject belongs to (the between-subject factor). A subject with a blank
# anywhere is left out entirely, because each subject needs every measurement.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
levels = ["Week 0", "Week 1", "Week 2", "Week 3"]
groups = ["Onboarding"]
factor_name = "Week" or "Within-subject factor"
subject_name = "User" or "Subject"
data[levels] = data[levels].apply(pd.to_numeric, errors="coerce")
data = data.dropna(subset=levels + groups).reset_index(drop=True)
y = data[levels].to_numpy(float)
n, k = y.shape  # subjects, conditions

## Descriptive statistics

In [ ]:
# Every factor and every combination of factors, levels in the order they appear.
long = data.melt(id_vars=groups, value_vars=levels, var_name=factor_name, value_name="value")
factors = [*groups, factor_name]
for name in factors:
    order = levels if name == factor_name else data[name].unique()
    long[name] = pd.Categorical(long[name], categories=order)
pieces = []
for size in range(1, len(factors) + 1):
    for chosen in combinations(factors, size):
        summary = long.groupby(list(chosen), observed=True)["value"].agg(
            n="count", mean="mean", SD="std")
        summary.index = [" * ".join(map(str, np.atleast_1d(key))) for key in summary.index]
        summary.insert(0, "model", " * ".join(chosen))
        pieces.append(summary)
descriptive = pd.concat(pieces)
descriptive["SE"] = descriptive["SD"] / np.sqrt(descriptive["n"])
descriptive = descriptive.assign(**{
    "mean - SD": descriptive["mean"] - descriptive["SD"],
    "mean + SD": descriptive["mean"] + descriptive["SD"],
    "mean - SE": descriptive["mean"] - descriptive["SE"],
    "mean + SE": descriptive["mean"] + descriptive["SE"],
})[["model", "n", "mean", "SD", "mean - SD", "mean + SD", "SE", "mean - SE", "mean + SE"]]
print(descriptive.to_string())

## Test for homogeneity of variance

In [ ]:
# Between the groups, separately for each condition. Bartlett skips a group with one
# subject or no spread; Levene's original form centres on each group's mean.
members = data.groupby(groups, sort=False).indices  # group -> its subjects' rows
rows = []
for j, level in enumerate(levels):
    samples = [y[index, j] for index in members.values()]
    usable = [s for s in samples if len(s) > 1 and s.var(ddof=1) > 0]
    bartlett = stats.bartlett(*usable)
    levene = stats.levene(*samples, center="mean")
    rows.append([bartlett.statistic, len(usable) - 1, bartlett.pvalue, levene.statistic,
                 len(samples) - 1, n - len(samples), levene.pvalue])
homogeneity = pd.DataFrame(rows, index=levels, columns=["Bartlett chi-square", "df", "P",
                                                        "Levene F", "df1", "df2", "P "])
print()
print(homogeneity)

## The two halves of the design

In [ ]:
# Between subjects: each subject's mean over the conditions, compared across groups.
# Within subjects: the k - 1 contrasts among the conditions (Helmert contrasts scaled
# to length 1; any orthonormal set gives the same answer), each regressed on the same
# group model. In that regression the intercept is the condition effect, each group
# term is its interaction with the condition, and the residual is the within-subject
# error. Type III with sum-to-zero coding throughout, as the app uses.
terms = [" * ".join(c) for size in range(1, len(groups) + 1) for c in combinations(groups, size)]
formula = " * ".join(f"C(Q({name!r}), Sum)" for name in groups)
subjects = data[groups].copy()
# The response goes into the same frame, under a name no group column has: a formula
# reads a name from the frame first, so a group called "score" would stand in for it.
score = "score"
while score in groups:
    score += "_"
subjects[score] = y.mean(axis=1)
between = anova_lm(smf.ols(f"{score} ~ {formula}", data=subjects).fit(), typ=3)
between.index = ["Intercept", *terms, "Residual"]
between["sum_sq"] *= k  # a subject mean stands for k measurements

scores = y @ helmert(k).T  # n x (k - 1)
fits = [smf.ols(f"{score} ~ {formula}", data=subjects.assign(**{score: scores[:, j]})).fit()
        for j in range(k - 1)]
within = sum(anova_lm(fit, typ=3)["sum_sq"] for fit in fits)
within.index = [factor_name, *[f"{term} * {factor_name}" for term in terms], "Residual"]
within_df = anova_lm(fits[0], typ=3)["df"].to_numpy() * (k - 1)

## Mauchly's test of sphericity

In [ ]:
# On the covariance of the contrasts' residuals from the group model.
residuals = np.column_stack([fit.resid for fit in fits])
df_resid = fits[0].df_resid  # subjects minus groups
covariance = residuals.T @ residuals / df_resid
w, chi_square, chi_df, chi_p = mauchly(covariance, df_resid)
lower, gg, hfl, cm = epsilons(covariance, n, groups=n - df_resid)
sphericity = pd.DataFrame(
    [[w, chi_square, chi_df, chi_p, lower, gg, hfl, cm]], index=[factor_name],
    columns=["W", "chi-square", "df", "P", "lower bound", "Greenhouse-Geisser",
             "Huynh-Feldt-Lecoutre", "Chi-Muller"],
)
print()
print(sphericity.to_string())

## Analysis of variance table

In [ ]:
# Within-subject rows are tested against the within-subject error, five ways: each
# correction multiplies both degrees of freedom by its epsilon (Huynh-Feldt capped at
# 1). Between-subject rows are tested against how much subjects differ in each group.
corrections = {"Sphericity assumed": 1.0, "Greenhouse-Geisser": gg,
               "Huynh-Feldt-Lecoutre": min(hfl, 1.0), "Chi-Muller": min(cm, 1.0),
               "Lower bound": lower}
ss_error, df_error = within["Residual"], within_df[-1]
rows = []
for source, ss, df in zip(within.index[:-1], within.to_numpy()[:-1], within_df[:-1]):
    f_value = (ss / df) / (ss_error / df_error)
    for correction, epsilon in corrections.items():
        df1, df2 = df * epsilon, df_error * epsilon
        rows.append([source, correction, ss, df1, ss / df1, f_value,
                     stats.f.sf(f_value, df1, df2)])
for correction, epsilon in corrections.items():
    df2 = df_error * epsilon
    rows.append([f"Error ({factor_name})", correction, ss_error, df2, ss_error / df2,
                 np.nan, np.nan])
ss_subject, df_subject = between.loc["Residual", "sum_sq"], between.loc["Residual", "df"]
for source in terms:
    ss, df = between.loc[source, "sum_sq"], between.loc[source, "df"]
    f_value = (ss / df) / (ss_subject / df_subject)
    rows.append([source, "", ss, df, ss / df, f_value, stats.f.sf(f_value, df, df_subject)])
rows.append([f"{subject_name} (error)", "", ss_subject, df_subject, ss_subject / df_subject,
             np.nan, np.nan])
rows.append(["Total", "", ((y - y.mean()) ** 2).sum(), n * k - 1, np.nan, np.nan, np.nan])
anova = pd.DataFrame(rows, columns=["source", "correction", "SS", "df", "MS", "F", "P"])
print()
print(anova.to_string(index=False))

## Effect sizes

In [ ]:
# Partial eta² sets each effect against its own error stratum: SS / (SS + SS error),
# the within-subject error for the condition and its interactions, the subjects for a
# group effect. It carries the 95% interval that inverts the noncentral F at the
# sphericity-assumed F (Steiger 2004), written out in eta_squared_interval.py.
# Generalised eta² sets every effect against all the error strata at once -- SS / (SS +
# SS subjects + SS within error) -- so it compares across designs (Olejnik & Algina
# 2003, every factor treated as manipulated).
from eta_squared_interval import partial_eta_squared_interval


def size(value):
    """Cohen's (1988) reading of an eta²: 0.01 small, 0.06 medium, 0.14 large."""
    return ["negligible", "small", "medium", "large"][sum(value >= s for s in (0.01, 0.06, 0.14))]


strata = [(source, ss, df, ss_error, df_error)
          for source, ss, df in zip(within.index[:-1], within.to_numpy()[:-1], within_df[:-1])]
strata += [(source, between.loc[source, "sum_sq"], between.loc[source, "df"], ss_subject,
            df_subject) for source in terms]
rows = []
for source, ss, df, error, error_df in strata:
    lower, upper = partial_eta_squared_interval((ss / df) / (error / error_df), df, error_df, 0.95)
    rows.append([source, "partial eta²", ss / (ss + error), lower, upper])
    rows.append([source, "generalised eta²", ss / (ss + ss_subject + ss_error), np.nan, np.nan])
effect_sizes = pd.DataFrame(rows, columns=["effect", "measure", "value", "lower", "upper"])
effect_sizes["size"] = effect_sizes["value"].map(size)
print()
print(effect_sizes.to_string(index=False))

## Multiple comparisons

In [ ]:
# Every pair of conditions (all groups together), with a t statistic on the
# within-subject error term, then each P value adjusted for the number of pairs by
# the chosen procedures.
# Any of "Bonferroni", "Sidak" and "Holm", in the order to print them; [] for none.
procedures = []
means = y.mean(axis=0)
se_difference = np.sqrt(2 * (ss_error / df_error) / n)
pairs = list(combinations(range(k), 2))
difference = np.array([means[i] - means[j] for i, j in pairs])
t_value = difference / se_difference
p_raw = 2 * stats.t.sf(np.abs(t_value), df_error)
rows = []
for procedure in procedures:
    p_adjusted = multipletests(p_raw, method=procedure.lower())[1]
    for (i, j), d, t, p in zip(pairs, difference, t_value, p_adjusted):
        rows.append([procedure, levels[i], levels[j], means[i], means[j], d, se_difference, t, p])
comparisons = pd.DataFrame(rows, columns=["method", "level 1", "level 2", "mean 1", "mean 2",
                                          "difference", "SE", "t", "P"])
print()
print(comparisons.to_string(index=False))

## Least squares means

In [ ]:
# Each mean averages the cell means with equal weights, and its standard error is
# sqrt(v * sum(1/n)) over the number of cells averaged, with a 95% interval on N - G
# degrees of freedom. For a group's mean, v is the subjects' error mean square. For a
# condition's or a cell's mean, v is the spread at that condition pooled across the
# groups, as a multivariate (unstructured) model estimates it; this needs no sphericity.
# The source sheet uses the subjects' term for those too, which overstates them.
group = groups[0]
group_levels = list(data[group].unique())
cell = {(g, c): y[(data[group] == g).to_numpy(), j]
        for g in group_levels for j, c in enumerate(levels)}
ms_subject = ss_subject / df_subject
spread = {c: sum(((cell[g, c] - cell[g, c].mean()) ** 2).sum() for g in group_levels) / df_subject
          for c in levels}
critical = stats.t.ppf(0.975, df_subject)


def lsmean(chosen, variance):
    mean = np.mean([values.mean() for values in chosen])
    se = np.sqrt(variance * sum(1 / len(values) for values in chosen)) / len(chosen)
    return [sum(len(values) for values in chosen), mean, se, mean - critical * se,
            mean + critical * se]


rows = [[group, g, "", *lsmean([cell[g, c] for c in levels], ms_subject)] for g in group_levels]
rows += [[factor_name, "", c, *lsmean([cell[g, c] for g in group_levels], spread[c])]
         for c in levels]
rows += [[f"{group} * {factor_name}", g, c, *lsmean([cell[g, c]], spread[c])]
         for g in group_levels for c in levels]
lsmeans = pd.DataFrame(rows, columns=["model", group, factor_name, "n", "LS mean", "SE",
                                      "lower", "upper"])
print()
print(lsmeans.to_string(index=False))

## Simple main effects

In [ ]:
# One factor tested separately at each level of the other, then Tukey's comparisons
# inside each level, on the standard error of a difference of two cell means. Groups
# at one condition are compared between subjects: the variance of one observation
# pools both error terms, (MS subjects + (k - 1) MS within) / k, on Satterthwaite's
# degrees of freedom, kept fractional. Conditions within one group are compared
# within subjects, against the within-subject error.
ms_within = ss_error / df_error
pooled = (ms_subject + (k - 1) * ms_within) / k
satterthwaite = (k * pooled) ** 2 / (ms_subject ** 2 / df_subject
                                     + ((k - 1) * ms_within) ** 2 / df_error)


def simple_main_effects(outer_levels, inner_levels, values, ms_err, ss_err, df_err):
    means = pd.DataFrame([[values(o, i).mean() for i in inner_levels] for o in outer_levels],
                         index=outer_levels, columns=inner_levels)
    counts = pd.DataFrame([[len(values(o, i)) for i in inner_levels] for o in outer_levels],
                          index=outer_levels, columns=inner_levels)
    tests, pairs = [], []
    for o in outer_levels:
        m, size = means.loc[o].to_numpy(), counts.loc[o].to_numpy()
        ss = (size * (m - (size * m).sum() / size.sum()) ** 2).sum()
        df = len(inner_levels) - 1
        f_value = (ss / df) / ms_err
        tests += [[o, "effect", ss, df, ss / df, f_value, stats.f.sf(f_value, df, df_err)],
                  [o, "Error", ss_err, df_err, ms_err, np.nan, np.nan]]
        for a, b in combinations(range(len(inner_levels)), 2):
            se = np.sqrt(ms_err * (1 / size[a] + 1 / size[b]))
            difference = abs(m[a] - m[b])
            statistic = difference / se
            pairs.append([o, inner_levels[a], inner_levels[b], m[a], m[b], difference, se,
                          statistic, stats.studentized_range.sf(statistic * np.sqrt(2),
                                                                len(inner_levels), df_err)])
    return {
        "means": means, "n": counts,
        "replications": pd.DataFrame([[counts.to_numpy().min(), counts.to_numpy().max()]],
                                     columns=["min", "max"]),
        "tests": pd.DataFrame(tests, columns=["level", "source", "SS", "df", "MS", "F", "P"]),
        "pairs": pd.DataFrame(pairs, columns=["level", "level 1", "level 2", "mean 1",
                                              "mean 2", "difference", "SE", "statistic", "P"]),
    }


simple = {
    "within": simple_main_effects(levels, group_levels, lambda c, g: cell[g, c],
                                  pooled, np.nan, satterthwaite),
    "between": simple_main_effects(group_levels, levels, lambda g, c: cell[g, c],
                                   ms_within, ss_error, df_error),
}
for outer, result in simple.items():
    print(f"\nSimple main effects within each level of the {outer}-subject factor")
    print(result["means"])
    print(result["n"])
    print(result["tests"].to_string(index=False))
    print(result["pairs"].to_string(index=False))

## What the result says

Each user was measured four times. F compares an effect with the noise, and the P value is the chance of an effect at least this large if there were none.

The overall difference between the groups is not significant: for Onboarding, F = 0.52 and P = 0.486. The benefit is in the interaction, Onboarding * Week, which asks whether the groups change differently over time: F = 58.8, with a P value far below 0.01.

The tests inside each week show when. At week 0 the groups are alike, 11.45 against 11.05 minutes a day (P = 0.699). At week 3 the old onboarding is at 4.88 minutes and the new one at 8.5, with P = 0.004. The new onboarding keeps users active for longer.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.